## tl;dr

- 范围：`智慧学习部指标汇总宽表` 最新 `dtdate=2026-08-29`、`季节财年=20272`，共 405 行。
- 新生：正基数 1,257 人次，按默认 30 人次门槛可 100% 自动覆盖；72.24% 在去校区层获得历史，5.09% 需要去产品宽范围回退，无人工缺口。按校区分类合计取整后预计剩余退班 20 人次。
- 老生：正式试算被 ADR-0122 的严格补集勾稽阻断。193 条正基数明细中 170 条不通过，影响 7,568 / 8,966 人次；物理老生过程分子合计 54，而严格补集应为 418。任何老生预计退班结果只能视为诊断值，不能发布。
- 粒度：发现 76 行业务维度不完整，但当前新老生基数均为 0；其中两行落在同一全空业务键。当前不影响人数结果，进入正式预测输入前仍应排除或明确归类。

## Context & Methods

本笔记本验证 ADR-0123 至 ADR-0130 的退班率回退覆盖率。输入由 `guancli ds preview` 从数据集 `l932b728584e3496685341a5` 只读导出到临时 CSV。

### Key Assumptions

- 历史窗口使用宽表当前行所带的上一财年同期过程值和最终值。
- 新生、老生分别独立回退；同一分类的过程率和最终率必须同层。
- 每层先汇总分子、分母；两项分母均须达到 30 人次，且四项历史字段完整。
- 当前分类基数为 0 的行不需要退班率，单独标记，不计入正基数覆盖率。
- 老生严格补集勾稽不通过时，老生结果整体阻断；脚本输出的物理字段诊断值不得用于正式预测。

## Data

In [ ]:
from pathlib import Path
import sys
import pandas as pd

workspace = Path.cwd()
if workspace.name == 'analysis':
    workspace = workspace.parent
sys.path.insert(0, str(workspace))

from analysis.withdrawal_fallback_validation import (
    CLASS_FIELDS, assign_fallbacks, load_data, profile_quality,
    rounded_totals, summarize_assignments,
)

input_csv = Path('/private/tmp/fy27_autumn_withdrawal_latest.csv')
threshold = 30
source_sql = workspace / 'analysis/sql/fy27_autumn_withdrawal_latest.sql'
data = load_data(input_csv, source_sql)
data.shape

In [ ]:
quality = profile_quality(data)
pd.DataFrame([quality]).T.rename(columns={0: '结果'})

## Results

In [ ]:
assignments = pd.concat(
    [assign_fallbacks(data, student_class, threshold) for student_class in CLASS_FIELDS],
    ignore_index=True,
)
coverage = pd.DataFrame(summarize_assignments(assignments))
coverage['分类内基数覆盖率'] = coverage['分类内基数覆盖率'].map(
    lambda value: '' if pd.isna(value) else f'{value:.2%}'
)
coverage[['退班分类', '回退层级', '明细数', '当前基数', '分类内基数覆盖率', '预计剩余退班人次_未取整']]

In [ ]:
pd.DataFrame(rounded_totals(assignments))

In [ ]:
affected_share = (
    quality['old_base_affected_by_complement_failure'] / quality['old_base_total']
)
pd.DataFrame({
    '校验': ['老生过程补集失败影响'],
    '失败正基数明细': [f"{quality['positive_old_base_complement_failure_rows']} / {quality['positive_old_base_rows']}"],
    '受影响老生基数': [f"{quality['old_base_affected_by_complement_failure']:.0f} / {quality['old_base_total']:.0f}"],
    '基数影响率': [f'{affected_share:.2%}'],
    '物理老生过程分子': [quality['process_old_physical_sum']],
    '严格补集应得分子': [quality['process_old_complement_sum']],
})

## Takeaways

1. 新生侧证明逐级回退可以显著降低人工量：正基数全部获得自动结果，只有 5.09% 基数进入“去产品”宽范围回退。
2. 默认 30 人次门槛下没有任何新生正基数明细能直接使用精确层，这不是失败，而是说明精确维度样本普遍偏小；去校区是主要有效层。
3. 老生侧当前不能评价回退机制优劣。应先修复 `老生同期退班人次-过程值`，刷新宽表最新分区并重新执行本笔记本。
4. 不完整业务维度行当前基数为零，对本次人数覆盖率无影响，但正式预测输入仍需明确排除，避免未来出现带人数的空键。